<a href="https://colab.research.google.com/github/austraraptor/2025_UNAP/blob/main/Ejercicios_Transformaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicios de la Sesión 3: Transformaciones de lazos
**Computación Paralela 2026-II, Universidad Nacional del Altiplano**

**Nombre:** David León Callohuanca Condori **Código:** 230928



Donde veas `COMPLETAR`, reemplázalo por tu código. Los datos son **valores ilustrativos**.

In [ ]:

import os, time, random
import numpy as np
from multiprocessing import Pool

class _Completar:
    def __repr__(self): return "COMPLETAR"
    def _propaga(self, *a): return self
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _propaga
    __truediv__ = __rtruediv__ = __pow__ = __rpow__ = __neg__ = __getitem__ = _propaga
COMPLETAR = _Completar()

def _tiene_completar(x):
    if isinstance(x, _Completar): return True
    if isinstance(x, np.ndarray): return x.dtype == object and any(_tiene_completar(v) for v in x.ravel())
    if isinstance(x, (list, tuple)): return any(_tiene_completar(v) for v in x)
    return False

def comprobar(nombre, original, transformado):
    """Compara el resultado original con el transformado."""
    if _tiene_completar(transformado):
        print(f"✏️ {nombre}: falta completar. Reemplaza COMPLETAR por tu código y vuelve a ejecutar.")
        return
    a, b = np.asarray(original, dtype=float), np.asarray(transformado, dtype=float)
    if a.shape == b.shape and np.allclose(a, b):
        print(f"✅ {nombre}: mismo resultado. La transformación es válida.")
    else:
        print(f"❌ {nombre}: el resultado CAMBIÓ. Revisa tu transformación.")

PROVINCIAS = ["Puno", "Azángaro", "Carabaya", "Chucuito", "El Collao", "Huancané", "Lampa",
              "Melgar", "Moho", "San Antonio de Putina", "San Román", "Sandia", "Yunguyo"]
rng = np.random.default_rng(2026)
print("✅ Listo. Núcleos disponibles:", os.cpu_count())

✅ Listo. Núcleos disponibles: 2


---
## Ejercicio 1: Ingreso y acumulado por provincia
```python
for p in range(1, 13):
    t = prod[p] * precio[p]
    ingreso[p] = t - costo[p]
    acum[p] = acum[p-1] + ingreso[p]
```

In [ ]:
prod   = list(rng.integers(80, 200, 13).astype(float))
precio = list(rng.integers(1, 4, 13).astype(float))
costo  = list(rng.integers(50, 150, 13).astype(float))

# ----- ORIGINAL (el lazo de la diapositiva) -----
ingreso = [0.0] * 13
acum    = [0.0] * 13
for p in range(1, 13):
    t = prod[p] * precio[p]
    ingreso[p] = t - costo[p]
    acum[p] = acum[p-1] + ingreso[p]

ingreso_orig, acum_orig = ingreso[:], acum[:]
print("Acumulado final:", acum_orig[-1])

Acumulado final: 2198.0


**1. ¿Qué dependencias tiene el lazo y cuáles son falsas?**

*Respuesta:*
**Dependencia verdadera (RAW)**:El lazo presenta las siguientes dependencias: acum[p] = acum[p-1] + ingreso[p], porque el valor de acum[p] depende del resultado calculado en la iteración anterior (acum[p-1]). Esta dependencia es real y no se puede eliminar directamente.

**Dependencia falsa:** la variable escalar t se reutiliza en cada iteración. Esto puede generar una dependencia de salida (WAW) o antidependencia (WAR) al reutilizar el mismo espacio de almacenamiento, aunque no existe una dependencia real entre los cálculos de ingresos de distintas provincias.


**2. ¿Qué transformaciones aplicarías?**

*Respuesta:*
Se aplicarian las siguientes transformaciones:

Expansión escalar: separar el cálculo de t para evitar que su reutilización genere dependencias falsas.

Distribución de lazos: dividir el lazo original en dos lazos independientes.
Paralelización DOALL: paralelizar el primer lazo, porque cada provincia calcula su ingreso independientemente.

Mantener secuencial el segundo lazo: conservar el cálculo del acumulado porque existe una dependencia verdadera entre iteraciones consecutivas.

**3. Escribe el código transformado.** Lazo A con expansión escalar y lazo B secuencial.




In [ ]:
t       = [0.0] * 13
ingreso = [0.0] * 13
acum    = [0.0] * 13

# Lazo A: DOALL
for p in range(1, 13):
    t[p] = prod[p] * precio[p]
    ingreso[p] = t[p] - costo[p]

# Lazo B: secuencial
for p in range(1, 13):
    acum[p] = acum[p-1] + ingreso[p]

comprobar("Ingreso", ingreso_orig, ingreso)
comprobar("Acumulado", acum_orig, acum)

✅ Ingreso: mismo resultado. La transformación es válida.
✅ Acumulado: mismo resultado. La transformación es válida.


**Ahora el lazo A con `Pool.map`:** cada proceso calcula una provincia. Dentro de la función, `t` es local: esa es la expansión escalar.

In [6]:
def ingreso_provincia(p):
    t = prod[p] * precio[p]
    return t - costo[p]

ingreso = [0.0] * 13
with Pool(4) as pool:
    ingreso[1:13] = pool.map(ingreso_provincia, range(1, 13))   # Lazo A en paralelo

acum = [0.0] * 13
for p in range(1, 13):                                          # Lazo B secuencial
    acum[p] = acum[p-1] + ingreso[p]

comprobar("Ingreso con Pool", ingreso_orig, ingreso)
comprobar("Acumulado con Pool", acum_orig, acum)

✅ Ingreso con Pool: mismo resultado. La transformación es válida.
✅ Acumulado con Pool: mismo resultado. La transformación es válida.


---
## Ejercicio 2: Rendimiento promedio de Puno en paralelo
```python
# registros: 520 tuplas (provincia, rendimiento)
suma = 0
n = 0
for prov, rend in registros:
    suma = suma + rend
    n = n + 1
promedio = suma / n
```

In [7]:
# 13 provincias x 4 cultivos x 10 años = 520 registros
registros = [(prov, round(float(rng.uniform(8, 25)), 1)) for prov in PROVINCIAS for _ in range(40)]

# ----- ORIGINAL (el lazo de la diapositiva) -----
suma = 0
n = 0
for prov, rend in registros:
    suma = suma + rend
    n = n + 1
promedio = suma / n

promedio_orig = promedio
print("Registros:", n, "  Promedio real:", round(promedio_orig, 3))

Registros: 520   Promedio real: 16.57


**1. ¿Por qué el lazo no es DOALL?**

*Respuesta:* No es DOALL por que existe dependencia RAW en el for mas expecificamrnte en:

    suma = suma + rend

    n = n + 1
promedio = suma / n

**Dependencia en suma:** cada iteración necesita el valor de suma calculado en la iteración anterior.

**Dependencia en n:** cada iteración necesita el valor de n incrementado anteriormente.

**2. Reparte los 520 registros en 4 procesos: ¿qué debe devolver cada proceso?**

*Respuesta:* Cada proceso debe devolver dos valores: la suma parcial de los rendimientos y el número de registros procesados. Como los tramos tienen diferentes tamaños (160, 120, 120 y 120), se suman los resultados parciales y se divide la suma total entre los 520 registros para obtener el promedio general.

Los 4 tramos se arman por grupos de provincias, así que **no tienen el mismo tamaño**:

In [9]:
cortes = [0, 160, 280, 400, 520]          # 4 provincias, 3, 3 y 3
tramos = [registros[cortes[i]:cortes[i+1]] for i in range(4)]
print("Registros por tramo:", [len(t) for t in tramos])

Registros por tramo: [160, 120, 120, 120]


**3. Escribe el código con `Pool.map`.**

In [10]:
def parcial(tramo):
    s = sum(r for _, r in tramo)                 # suma de los rendimientos del tramo
    return s, len(tramo)          # (suma, conteo)

with Pool(4) as pool:
    partes = pool.map(parcial, tramos)
suma = sum(p[0] for p in partes)
n    = sum(p[1] for p in partes)
promedio = suma / n

comprobar("Reducción de sumas y conteos", promedio_orig, promedio)

✅ Reducción de sumas y conteos: mismo resultado. La transformación es válida.


**¿Qué error habría si cada proceso devuelve su promedio?** Ejecuta y compara.

In [12]:
def promedio_tramo(tramo):
    return sum(r for _, r in tramo) / len(tramo)

with Pool(4) as pool:
    promedios = pool.map(promedio_tramo, tramos)
promedio_de_promedios = sum(promedios) / 4

print("Promedio real        :", round(promedio_orig, 3))
print("Promedio de promedios:", round(promedio_de_promedios, 3))
comprobar("Promedio de promedios", promedio_orig, promedio_de_promedios)

Promedio real        : 16.57
Promedio de promedios: 16.581
❌ Promedio de promedios: el resultado CAMBIÓ. Revisa tu transformación.


*Respuesta:* El error es que cada tramo tiene diferente cantidad de registros. Al promediar los cuatro promedios, todos reciben el mismo peso, lo que altera el resultado. Por eso, se debe sumar los rendimientos y los conteos parciales para obtener el promedio general correcto: 16.570.

---
## Ejercicio 3: Humedad del suelo por provincia y mes
```python
for m in range(1, 12):         # mes
    for p in range(13):        # provincia
        H[p][m] = H[p][m-1] * 0.5 + lluvia[p][m]
```

In [11]:
lluvia = np.round(rng.uniform(0, 150, (13, 12)), 1)     # mm, 13 provincias x 12 meses

# ----- ORIGINAL (el lazo de la diapositiva) -----
H = np.zeros((13, 12))
H[:, 0] = lluvia[:, 0]
for m in range(1, 12):         # mes
    for p in range(13):        # provincia
        H[p][m] = H[p][m-1] * 0.5 + lluvia[p][m]

H_orig = H.copy()
print("Humedad de Puno en diciembre:", round(H_orig[0][11], 2))

Humedad de Puno en diciembre: 129.95


**1. ¿Cuál es el vector distancia, en orden (provincia, mes)?**

*Respuesta:* El vector distancia es (0, 1), porque el cálculo depende de la humedad de la provincia en el mes anterior, no de otra provincia.

**2. ¿Qué lazo es paralelo: el externo o el interno?**

*Respuesta:* El lazo externo (mes) es secuencial por la dependencia entre meses, mientras que el lazo interno (provincia) es paralelo, porque cada provincia se calcula independientemente de las demás.

**3. Transforma el código para repartir con un solo `Pool.map`.** Intercambia los lazos: provincias afuera y meses adentro.

In [13]:
def humedad(fila):
    h = [fila[0]]
    for m in range(1, len(fila)):
        h.append(h[m-1] * 0.5 + fila[m])
    return h

funcion = humedad            # ¿qué función se reparte?
datos   = lluvia            # ¿sobre qué datos? (una fila por provincia)

if _tiene_completar([funcion, datos]):
    H = COMPLETAR
else:
    with Pool(4) as pool:
        H = pool.map(funcion, datos)

comprobar("Intercambio de lazos", H_orig, H)

✅ Intercambio de lazos: mismo resultado. La transformación es válida.


---
## Ejercicio 4: Rendimiento de 13 provincias en 10 años
```python
for p in range(13):
    for a in range(10):
        rend[p][a] = prod[p][a] / area[p][a]
```

In [14]:
prod = rng.uniform(50, 300, (13, 10))     # toneladas
area = rng.uniform(5, 30, (13, 10))       # hectáreas

# ----- ORIGINAL (el lazo de la diapositiva) -----
t0 = time.perf_counter()
rend = np.zeros((13, 10))
for p in range(13):
    for a in range(10):
        rend[p][a] = prod[p][a] / area[p][a]
t_for = time.perf_counter() - t0

rend_orig = rend.copy()
print(f"Con lazos for: {t_for*1000:.3f} ms")

Con lazos for: 0.384 ms


**1. ¿Es DOALL? ¿Por qué?**

*Respuesta:* Si Es DOALL por que ninguna variable depende de otra.

**2. Escribe la versión con NumPy, sin lazos.**

In [15]:
t0 = time.perf_counter()
rend = prod / area              # prod y area: arreglos de 13 x 10
t_np = time.perf_counter() - t0

comprobar("Vectorización", rend_orig, rend)
if not _tiene_completar(rend):
    print(f"Con NumPy: {t_np*1000:.3f} ms")

✅ Vectorización: mismo resultado. La transformación es válida.
Con NumPy: 0.179 ms


**3. ¿Conviene usar Pool para estas 130 divisiones? ¿Por qué?** Ejecuta, mira los tiempos y responde.

In [16]:
def rend_provincia(p):
    return prod[p] / area[p]

t0 = time.perf_counter()
with Pool(4) as pool:
    rend_pool = pool.map(rend_provincia, range(13))
t_pool = time.perf_counter() - t0

comprobar("Con Pool", rend_orig, rend_pool)
print(f"Con lazos for: {t_for*1000:8.3f} ms")
print(f"Con Pool     : {t_pool*1000:8.3f} ms")

✅ Con Pool: mismo resultado. La transformación es válida.
Con lazos for:    0.384 ms
Con Pool     :   57.355 ms


*Respuesta:* No conviene utilizar Pool para estas 130 divisiones, porque el tiempo de creación y distribución de procesos supera el tiempo de ejecución. Como se observa, con for tarda 0.384 ms, mientras que con Pool tarda 57.355 ms. Por tanto, es más eficiente utilizar la vectorización de NumPy para este caso.